In [1]:
from pyspark.sql import SparkSession
from config.settings import settings
import os

In [2]:
spark = SparkSession.builder \
    .appName("YelpExploration") \
    .master("local[*]")        \
    .config("spark.driver.memory", "1500m") \
    .config("spark.driver.maxResultSize", "512m") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.files.maxPartitionBytes", "128m") \
    .getOrCreate()

26/02/27 19:37:21 WARN Utils: Your hostname, Rahuls-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.116 instead (on interface en0)
26/02/27 19:37:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 19:37:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Schema

In [3]:
spark.sparkContext.setJobDescription("BIZ - read + infer schema")
biz = spark.read \
    .option("inferSchema", "true") \
    .option("samplingRatio", "1.0") \
    .json(settings.yelp.BUSINESS_JSON_PATH)
print("Business DataFrame created")

Business DataFrame created


26/02/27 19:37:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
spark.sparkContext.setJobDescription("REV - read + infer schema")
rev = spark.read \
    .option("inferSchema", "true") \
    .option("samplingRatio", "1.0") \
    .json(settings.yelp.REVIEW_JSON_PATH)
print("Review DataFrame created")

Review DataFrame created


In [5]:
output_path = "exploration/schema_exploration.txt"

with open(output_path, "w") as f:
    f.write("=== BUSINESS SCHEMA ===\n")
    schema_str = biz._jdf.schema().treeString()
    f.write(schema_str + "\n")
    spark.sparkContext.setJobDescription("BIZ - count rows")
    f.write(f"Total rows: {biz.count()}\n\n")

    # Review
    f.write("=== REVIEW SCHEMA ===\n")
    schema_str = rev._jdf.schema().treeString()
    f.write(schema_str + "\n")
    spark.sparkContext.setJobDescription("REV - count rows")
    f.write(f"Total rows: {rev.count()}\n")

spark.sparkContext.setJobDescription(None)
print(f"Schema written to: {os.path.abspath(output_path)}")


Schema written to: /Users/rahul/Learning/Coding/yelp-streaming-intelligence/exploration/schema_exploration.txt


## Value Distribution

In [6]:
OUTPUT_FILE = "exploration/yelp_distinct_values.txt"

def explore_column(f, df, col, threshold=50):
    distinct_count = df.select(col).distinct().count()
    f.write(f"\n{'─'*50}\n")
    f.write(f"  Column         : {col}\n")
    f.write(f"  Distinct count : {distinct_count}\n")

    if distinct_count <= threshold:
        rows = df.select(col).distinct().orderBy(col).collect()
        f.write(f"  Distinct values:\n")
        for row in rows:
            f.write(f"    {row[0]}\n")
    else:
        f.write(f"  (High cardinality — top 20 by frequency)\n")
        rows = df.groupBy(col).count().orderBy("count", ascending=False).limit(20).collect()
        f.write(f"  {'Value':<50} {'Count'}\n")
        for row in rows:
            f.write(f"    {str(row[0]):<50} {row[1]}\n")


In [7]:
business_flat_cols = [
    "address",          # string
    "business_id",      # string - primary
    "name",             # string
    "city",             # string
    "state",            # string
    "postal_code",      # string
    "latitude",         # float
    "longitude",        # float
    "categories",       # comma seperated strings
    "stars",            # float
    "review_count",     # integer
    "is_open"           # integer 0 and 1 -> bool
]

business_attribute_cols = [
    "attributes.Alcohol",                           # string -> bool 
    "attributes.BYOB",                              # string -> bool 
    "attributes.BikeParking",                       # string -> bool 
    "attributes.BusinessAcceptsBitcoin",            # string -> bool 
    "attributes.BusinessAcceptsCreditCards",        # string -> bool 
    "attributes.BusinessParking",                   # dictionary  {'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}
    "attributes.DogsAllowed",                       # string -> bool
    "attributes.DriveThru",                         # string -> bool
    "attributes.GoodForDancing",                    # string -> bool
    "attributes.GoodForKids",                       # string -> bool
    "attributes.HasTV",                             # string -> bool
    "attributes.Music",                             # dictionary {'dj': False, 'background_music': False, 'no_music': False, 'jukebox': False, 'live': False, 'video': False, 'karaoke': False}
    "attributes.NoiseLevel",                        # string
    "attributes.Open24Hours",                       # string -> bool
    "attributes.OutdoorSeating",                    # string -> bool
    "attributes.RestaurantsDelivery",               # string -> bool
    "attributes.RestaurantsGoodForGroups",          # string -> bool
    "attributes.RestaurantsPriceRange2",            # integer 1,2,3,4?
    "attributes.Smoking",                           # string
    "attributes.WheelchairAccessible",              # string -> bool
    "attributes.WiFi"                               # string
]

# string? or timestamp?
business_hours_cols = [
    "hours.Monday", 
    "hours.Tuesday", 
    "hours.Wednesday",
    "hours.Thursday", 
    "hours.Friday", 
    "hours.Saturday", 
    "hours.Sunday"
]

review_cols = [
    "review_id",            # string
    "business_id",          # string - primary
    "stars",                # float
    "date",                 # timestamp
    "text"                  # string
]

In [8]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for col in business_flat_cols:
        spark.sparkContext.setJobDescription(f"Exploring BIZ {col}")
        explore_column(f, biz, col)

    for col in business_attribute_cols:
        spark.sparkContext.setJobDescription(f"Exploring BIZ {col}")
        explore_column(f, biz, col)

    for col in business_hours_cols:
        spark.sparkContext.setJobDescription(f"Exploring BIZ {col}")
        explore_column(f, biz, col)

    for col in review_cols:
        spark.sparkContext.setJobDescription(f"Exploring REV {col}")
        explore_column(f, rev, col)

spark.sparkContext.setJobDescription(None)
print(f"Done. Output written to: {os.path.abspath(OUTPUT_FILE)}")

26/02/27 19:38:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/27 19:38:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/27 19:38:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/27 19:38:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/27 19:38:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/27 19:38:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/27 19:38:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/27 19:38:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/02/27 19:38:20 WARN RowBasedKeyValueBatch: Calling spill() on

Done. Output written to: /Users/rahul/Learning/Coding/yelp-streaming-intelligence/exploration/yelp_distinct_values.txt
